<img src="https://raw.githubusercontent.com/carubbi/MQ/main/notebooks/assets/imgs/unifor-logo.png" width="400">
<br>
<b>
<font size="6" face="arial" color="blue">
    Graduação em Ciência da Computação
</font>
</b>
<br>
<b>
<font size="4" face="arial">
    Disciplina: Métodos Quantitativos em Computação
</font>
</b>

**Orientador: Prof. Me. Ricardo Carubbi** <br>
*Docente da Graduação e Pós-Graduação em Ciência de Dados e Inteligência Artificial*<br>
*Laboratório de Ciência de Dados e Inteligência Artificial*<br>
*Universidade de Fortaleza*<br>

Lattes: http://lattes.cnpq.br/5738786447903616 |
GitHub: https://github.com/carubbi/

[Unifor.br](https://unifor.br/) | [Instagram](https://www.instagram.com/uniforcomunica/?hl=pt-br) | [Facebook](https://www.facebook.com/uniforoficial/) | [Twitter](https://www.facebook.com/uniforoficial/)  | [LinkedIn](https://www.linkedin.com/school/university-of-fortaleza/?originalSubdomain=pt) | [TV Unifor](https://www.unifor.br/tv-unifor) | [G1/Ensinando e Aprendendo](https://g1.globo.com/ce/ceara/especial-publicitario/unifor/ensinando-e-aprendendo/)

# AP2: Auditoria de modelos probabilísticos

## 1. Identificação

- **Disciplina:** Métodos Quantitativos em Computação (T199)
- **Turma:** T199-64
- **Integrantes:** Adriel Medeiros Lins
- **Característica qualitativa $B$ atribuída:** Qualidade da cozinha (`Kitchen Qual`), agrupada na AP1 em `Kitchen Qual Final`
- **Modelos candidatos:** Binomial, Poisson e Normal
- **Repositório:** <https://github.com/medeiros-cs/mq-ap>

## 2. Delimitação da investigação

### 2.1 Contexto

Na AP1, os dados de imóveis residenciais do Ames Housing foram preparados e explorados, e a qualidade da cozinha foi agrupada nas categorias finais `Fa/Po`, `TA`, `Gd` e `Ex`. Nesta etapa, essa característica é transformada em um evento binário, que dá origem às variáveis de três trilhas com modelos probabilísticos candidatos: Binomial e Poisson, para contagens do evento, e Normal, para o preço de venda dos imóveis em que ele ocorre. Cada modelo é confrontado com a variável correspondente, verificando tipo, suporte, mecanismo e pressupostos antes da caracterização empírica e da determinação dos parâmetros.

### 2.2 Pergunta de pesquisa

Os modelos probabilísticos atribuídos são adequados para representar as contagens de imóveis com cozinha de qualidade boa ou excelente e o preço de venda desses imóveis?

### 2.3 Objetivo da investigação

Avaliar criticamente a adequação dos modelos Binomial, de Poisson e Normal a partir dos dados observados no Ames Housing, confrontando-os com as propriedades teóricas e os pressupostos de cada modelo.

### 2.4 Atribuição

*Registro da atribuição exigido pelo enunciado. A descrição do evento em linguagem natural integra a Etapa 1.*

Registrada antes da análise e não substituída em razão dos resultados:

- **Característica:** qualidade da cozinha (`Kitchen Qual Final`), definida na AP1.
- **Evento binário:** "o imóvel tem cozinha de qualidade boa ou excelente", isto é, `Kitchen Qual Final` igual a `Gd` (Good) ou `Ex` (Excellent). As categorias `TA` (Typical/Average) e `Fa/Po` (Fair/Poor) formam o resultado complementar. O corte separa as cozinhas acima do padrão típico das demais.
- **Trilha Binomial:** número de imóveis com o evento em $n=20$ vendas.
- **Trilha Poisson:** número de vendas de imóveis com o evento em cada mês.
- **Trilha Normal:** preço de venda (`SalePrice`) dos imóveis com o evento.

## 3. Carregamento dos dados

Os dados são carregados de `data/processed/AmesHousing.csv`, produzido na AP1, sem redefinir a limpeza e as decisões registradas naquela etapa.

In [ ]:
# Importar as bibliotecas de manipulação de dados e de visualização
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

In [ ]:
# Carregar o dado pré-processado na AP1
df = pd.read_csv("../data/processed/AmesHousing.csv")
df.shape

**Tabela 1 - Dimensões do dado pré-processado.** O arquivo tem 2.930 vendas e 83 colunas: as 82 originais e `Kitchen Qual Final`, criada na AP1.

## 4. Delimitação da variável aleatória

*Etapa 1 do enunciado: Delimitação da variável aleatória.*

### 4.1 Unidade de análise e definição

*Etapa 1: unidade de análise; descrição da variável ou do evento em linguagem natural; definição da variável aleatória e de seus valores possíveis.*

#### Binarização da qualidade da cozinha

Cada imóvel vendido é a unidade de análise da trilha Binomial e funciona como um ensaio com dois resultados: a cozinha é boa ou excelente (sucesso, 1) ou não é (fracasso, 0).

In [ ]:
# Binarizar Kitchen Qual Final: 1 para cozinha Gd ou Ex, 0 para TA ou Fa/Po
df["Kitchen Gd or Ex"] = df["Kitchen Qual Final"].isin(["Gd", "Ex"]).astype(int)
pd.crosstab(df["Kitchen Qual Final"], df["Kitchen Gd or Ex"]).reindex(["Fa/Po", "TA", "Gd", "Ex"])

**Tabela 2 - Binarização da qualidade da cozinha.** As categorias `Gd` (1.160 imóveis) e `Ex` (205) recebem o valor 1, totalizando 1.365 imóveis com cozinha boa ou excelente; `TA` (1.494) e `Fa/Po` (71) recebem o valor 0, totalizando 1.565.

#### Trilha Binomial: imóveis com cozinha boa ou excelente em 20 vendas

Seja $X$ o número de imóveis com cozinha boa ou excelente entre $n=20$ vendas. Como cada imóvel é um único ensaio, os valores observados de $X$ exigem conjuntos reais de 20 vendas: as vendas são ordenadas por `Yr Sold`, `Mo Sold` e `Order` e divididas em grupos consecutivos de 20, de modo análogo a uma sequência de 20 e-mails observados. Cada grupo fornece uma observação de $X$. A ordem é cronológica apenas até o mês, pois a base registra o mês da venda, mas não o dia; dentro de cada mês, `Order` acompanha o código do imóvel (`PID`), que tende a reunir imóveis do mesmo bairro.

In [ ]:
# Formar grupos de 20 vendas consecutivas, ordenadas por ano, mês e Order, e contar o evento em cada grupo
vendas_ordenadas = df.sort_values(["Yr Sold", "Mo Sold", "Order"]).reset_index(drop=True)
n_grupos = len(vendas_ordenadas) // 20
vendas_agrupadas = vendas_ordenadas.iloc[: n_grupos * 20].copy()
vendas_agrupadas["grupo"] = vendas_agrupadas.index // 20 + 1

X = vendas_agrupadas.groupby("grupo")["Kitchen Gd or Ex"].sum().rename("X")
X.head()

**Tabela 3 - Primeiras observações de $X$.** Cada linha é um grupo de 20 vendas consecutivas e registra quantos desses imóveis têm cozinha boa ou excelente. As 2.930 vendas formam 146 grupos completos; as 10 vendas finais (as 2 últimas de junho e as 8 de julho de 2010) não completam um grupo e ficam fora desta trilha.

#### Trilha Poisson: vendas mensais de imóveis com cozinha boa ou excelente

Na trilha Poisson, a unidade de análise é o **mês de venda**, identificado por `Yr Sold` e `Mo Sold`. Seja $Y$ o número de vendas de imóveis com cozinha boa ou excelente em um mês, isto é, uma contagem de ocorrências do evento em um intervalo de tempo.

In [ ]:
# Contar, em cada mês, as vendas de imóveis com cozinha boa ou excelente
Y = df.groupby(["Yr Sold", "Mo Sold"])["Kitchen Gd or Ex"].sum().rename("Y")
Y.head()

**Tabela 4 - Primeiras observações de $Y$.** Cada linha é um mês de venda. A base cobre 55 meses, de janeiro de 2006 a julho de 2010; como a contagem parte de todas as vendas do mês, um mês sem vendas com o evento aparece com $Y=0$.

#### Trilha Normal: preço de venda dos imóveis com cozinha boa ou excelente

Na trilha Normal, a unidade de análise volta a ser o **imóvel vendido**, agora restrito aos 1.365 imóveis com cozinha boa ou excelente. Seja $W$ o preço de venda (`SalePrice`, em dólares) de um imóvel com o evento, uma medida em escala contínua, e não uma contagem.

In [ ]:
# Selecionar o preço de venda dos imóveis com cozinha boa ou excelente
W = df.loc[df["Kitchen Gd or Ex"] == 1, "SalePrice"].rename("W")
W.head()

**Tabela 5 - Primeiras observações de $W$.** Cada linha é um imóvel com cozinha boa ou excelente, identificado pelo índice da linha no arquivo, e registra o seu preço de venda em dólares.

### 4.2 Classificação e suporte

*Etapa 1: classificação como discreta ou contínua; suporte observado e suporte previsto pelo modelo.*

$X$ e $Y$ são variáveis aleatórias **discretas**: ambas contam ocorrências e assumem apenas valores inteiros não negativos. O suporte previsto pelo modelo Binomial é $\{0, 1, \ldots, 20\}$; o do modelo de Poisson é $\{0, 1, 2, \ldots\}$, sem limite superior.

$W$ é tratada como variável aleatória **contínua**: o preço é uma medida monetária, registrado em dólares inteiros, mas com escala fina o bastante para que o arredondamento seja irrelevante. O suporte previsto pelo modelo Normal é toda a reta real, $(-\infty, \infty)$.

In [ ]:
# Comparar o suporte observado das três variáveis
pd.DataFrame({
    "Mínimo observado": [X.min(), Y.min(), W.min()],
    "Máximo observado": [X.max(), Y.max(), W.max()],
    "Valores distintos observados": [X.nunique(), Y.nunique(), W.nunique()],
}, index=["X (Binomial)", "Y (Poisson)", "W (Normal)"])

**Tabela 6 - Suporte observado.** $X$ assume valores de 1 a 20, dentro do suporte $\{0, \ldots, 20\}$ do modelo Binomial e atingindo o limite natural de 20. $Y$ assume valores de 0 a 58. O valor 58 é apenas o maior observado, e não um limite: o modelo de Poisson admite qualquer inteiro não negativo, sem teto. Na realidade, porém, $Y$ tem um teto que muda a cada mês, o total de vendas do mês (de 8 a 122 nos 55 meses; em julho de 2010, com 8 vendas, $Y$ não poderia passar de 8). O modelo não contempla esse limite. $W$ vai de US\$ 59.000 a US\$ 755.000, com 706 valores distintos entre 1.365 imóveis, o que confirma a escala praticamente contínua. O preço é sempre positivo, enquanto o modelo Normal atribui probabilidade também a valores negativos.

### 4.3 Pergunta específica da investigação

*Etapa 1: formulação da pergunta específica da investigação.*

- **Trilha Binomial:** o modelo $\mathrm{Binomial}(20, p)$ representa adequadamente o número de imóveis com cozinha boa ou excelente entre 20 vendas consecutivas?
- **Trilha Poisson:** o modelo $\mathrm{Poisson}(\lambda)$ representa adequadamente o número mensal de vendas de imóveis com cozinha boa ou excelente?
- **Trilha Normal:** o modelo $\mathrm{Normal}(\mu, \sigma^2)$ representa adequadamente o preço de venda dos imóveis com cozinha boa ou excelente?

## 5. Compatibilidade conceitual inicial

*Etapa 2 do enunciado: Compatibilidade inicial do modelo atribuído (tipo da variável, suporte, mecanismo probabilístico, parâmetros e pressupostos essenciais).*

Antes de qualquer cálculo ou comparação gráfica, cada modelo é confrontado com as variáveis definidas. Os pontos marcados como "a examinar" indicam os pressupostos que dependem de verificação, sem antecipar parecer.

| Critério | Trilha Binomial: $X$ | Trilha Poisson: $Y$ | Trilha Normal: $W$ |
|---|---|---|---|
| **Tipo da variável** | Contagem discreta; compatível. | Contagem discreta; compatível. | Medida contínua; compatível. |
| **Suporte** | $\{0, \ldots, 20\}$, igual ao do modelo; compatível por construção. | Observado de 0 a 58; o modelo admite qualquer inteiro não negativo, mas $Y$ é limitado pelo total de vendas do mês; a examinar. | O modelo admite toda a reta real, inclusive valores negativos, mas o preço é sempre positivo; a examinar. |
| **Mecanismo** | 20 ensaios (vendas), cada um com dois resultados (cozinha boa ou excelente, ou não); compatível. | Ocorrências do evento contadas em um intervalo de tempo fixo (o mês); compatível com o mecanismo de Poisson. | O modelo Normal surge da soma de muitos efeitos pequenos e independentes. O preço resulta de muitos fatores (área, localização, idade, acabamento), mas eles tendem a agir de forma proporcional ao valor do imóvel, e não aditiva; a examinar. |
| **Parâmetros** | $n = 20$, fixado; $p$, probabilidade de cozinha boa ou excelente, a ser calibrada pela frequência relativa observada. | $\lambda$, número médio de vendas com o evento por mês, a ser calibrado pela média observada. | $\mu$ e $\sigma^2$, média e variância do preço, a serem calibradas pela média e pela variância amostrais. |
| **Pressupostos essenciais** | Probabilidade $p$ constante entre as vendas e independência entre elas: a examinar, pois vendas de imóveis novos e de um mesmo empreendimento tendem a ocorrer juntas no tempo. As seleções são feitas sem reposição em um conjunto finito de 2.930 vendas. | Taxa constante ao longo dos meses: a examinar, pois o mercado imobiliário é sazonal. Independência entre meses e igualdade entre média e variância: a examinar. | Simetria em torno da média e caudas leves: a examinar, pois na AP1 o preço de venda do conjunto completo mostrou forte assimetria à direita. Independência entre vendas: a examinar, pelo mesmo agrupamento de empreendimentos e bairros. |

## 6. Caracterização empírica

*Etapa 3 do enunciado: Descrição empírica.*

### 6.1 Observações válidas e ausentes

*Etapa 3: quantidade de observações válidas e valores ausentes.*

In [ ]:
# Verificar observações válidas e ausentes das colunas usadas nas três trilhas
colunas_usadas = ["Kitchen Qual Final", "Yr Sold", "Mo Sold", "SalePrice"]
pd.DataFrame({
    "Válidas": df[colunas_usadas].notna().sum(),
    "Ausentes": df[colunas_usadas].isna().sum(),
})

**Tabela 7 - Observações válidas e ausentes.** As quatro colunas usadas não têm valores ausentes nas 2.930 vendas. A trilha Binomial usa 2.920 vendas (146 grupos de 20), a trilha Poisson usa todas as 2.930, distribuídas em 55 meses, e a trilha Normal usa os 1.365 imóveis com cozinha boa ou excelente.

### 6.2 Medidas e distribuição de frequências

*Etapa 3: tabela de frequências relativas; média e variância observadas.*

#### Trilha Binomial

In [ ]:
# Construir a tabela de frequências absolutas e relativas de X
freq_X = X.value_counts().reindex(range(21), fill_value=0).to_frame("Frequência absoluta")
freq_X["Frequência relativa"] = (freq_X["Frequência absoluta"] / len(X)).round(3)
freq_X.index.name = "X"
freq_X

**Tabela 8 - Frequências de $X$.** Os 146 grupos concentram-se entre 6 e 13 imóveis com cozinha boa ou excelente, que reúnem 75% das observações (109 de 146). Os valores 0 e 17 não foram observados, e as caudas são longas: há grupos com apenas 1 ou 2 imóveis com o evento e grupos com 18, 19 e 20.

#### Trilha Poisson

In [ ]:
# Construir a tabela de frequências de Y em classes de 10 vendas
classes_Y = pd.cut(Y, bins=range(0, 70, 10), right=False)
freq_Y = classes_Y.value_counts().sort_index().to_frame("Frequência absoluta")
freq_Y["Frequência relativa"] = (freq_Y["Frequência absoluta"] / len(Y)).round(3)
freq_Y.index.name = "Y (vendas no mês)"
freq_Y

**Tabela 9 - Frequências de $Y$.** Como $Y$ assume muitos valores distintos em apenas 55 meses, as frequências são apresentadas em classes de 10 vendas. Quase dois terços dos meses (35 de 55) têm de 10 a 29 vendas com o evento; os 9 meses com 40 ou mais vendas formam uma cauda à direita.

#### Trilha Normal

In [ ]:
# Construir a tabela de frequências de W em classes de US$ 50 mil
classes_W = pd.cut(W, bins=range(50_000, 800_001, 50_000), right=False)
freq_W = classes_W.value_counts().sort_index().to_frame("Frequência absoluta")
freq_W["Frequência relativa"] = (freq_W["Frequência absoluta"] / len(W)).round(3)
freq_W.index.name = "W (preço de venda, US$)"
freq_W

**Tabela 10 - Frequências de $W$.** Como $W$ é contínua, as frequências são apresentadas em classes de US\$ 50 mil. Mais da metade dos imóveis (733 de 1.365, 54%) foi vendida entre US\$ 150 mil e US\$ 250 mil. À esquerda, a distribuição termina logo, com apenas 16 imóveis abaixo de US\$ 100 mil; à direita, as frequências diminuem devagar, e 33 imóveis passam de US\$ 450 mil, chegando a US\$ 755 mil.

#### Medidas descritivas das três trilhas

In [ ]:
# Calcular média, variância e demais medidas descritivas das três variáveis
variaveis = {"X (Binomial)": X, "Y (Poisson)": Y, "W (Normal)": W}
pd.DataFrame({
    nome: {
        "Observações": v.count(),
        "Média": v.mean(),
        "Variância": v.var(),
        "Desvio padrão": v.std(),
        "Mínimo": v.min(),
        "Mediana": v.median(),
        "Máximo": v.max(),
        "Assimetria": v.skew(),
    }
    for nome, v in variaveis.items()
}).style.format(precision=2)

**Tabela 11 - Medidas descritivas.** $X$ tem média 9,35 e variância 13,57 imóveis², com leve assimetria à direita (0,28). $Y$ tem média 24,82 e variância 206,56 vendas², com assimetria de 0,92. $W$ tem média de US\$ 229.834 e desvio padrão de US\$ 86.229 (a variância, de cerca de 7,44 × 10⁹ dólares², é pouco interpretável por estar em dólares ao quadrado). A média de $W$ supera a mediana (US\$ 213.000) e a assimetria é de 1,51, sinais de uma cauda longa à direita.

### 6.3 Representação gráfica

*Etapa 3: representação gráfica coerente com o tipo da variável; características relevantes para a auditoria (concentração, assimetria, caudas, excesso de zeros, limites naturais).*

#### Trilha Binomial

In [ ]:
# Construir o gráfico de barras das frequências relativas de X
fig, eixo = plt.subplots(figsize=(8, 4))
eixo.bar(freq_X.index, freq_X["Frequência relativa"], color="#4C72B0")
eixo.set_xlabel("Imóveis com cozinha boa ou excelente em 20 vendas (X)")
eixo.set_ylabel("Frequência relativa")
eixo.set_title("Distribuição de X em 146 grupos de 20 vendas")
eixo.set_xticks(range(21))
plt.show()

**Figura 1 - Distribuição de $X$.** A distribuição é unimodal, centrada perto de 9 imóveis, com leve assimetria à direita (coeficiente 0,28). Os valores observados vão de 1 a 20, alcançando o limite superior natural, com caudas longas dos dois lados que reforçam a dispersão elevada observada na Tabela 8.

#### Trilha Poisson

In [ ]:
# Construir o histograma das frequências relativas de Y
fig, eixo = plt.subplots(figsize=(8, 4))
eixo.hist(Y, bins=range(0, 65, 5), weights=[1 / len(Y)] * len(Y), color="#55A868", edgecolor="white")
eixo.set_xlabel("Vendas de imóveis com cozinha boa ou excelente no mês (Y)")
eixo.set_ylabel("Frequência relativa")
eixo.set_title("Distribuição de Y em 55 meses")
plt.show()

**Figura 2 - Distribuição de $Y$.** A distribuição é assimétrica à direita (coeficiente 0,92): a maioria dos meses tem entre 10 e 30 vendas com o evento, e poucos meses passam de 50. Há um único mês com $Y=0$ (julho de 2010, último mês da base, com apenas 8 vendas); portanto, não há excesso de zeros.

In [ ]:
# Calcular a média de Y por mês do ano para examinar a sazonalidade
Y.groupby(level="Mo Sold").mean().round(1).to_frame("Média de Y")

**Tabela 12 - Média de $Y$ por mês do ano.** As vendas com o evento concentram-se em maio, junho e julho (médias de 35,2, 48,4 e 41,8; a média de julho inclui julho de 2010, com $Y=0$) e caem no inverno (12,0 em dezembro e 12,6 em janeiro). Essa sazonalidade indica uma taxa que varia ao longo do ano e ajuda a explicar a variância elevada da Tabela 11. É uma característica central para o pressuposto de taxa constante.

#### Trilha Normal

In [ ]:
# Construir o histograma das frequências relativas de W e o boxplot correspondente
fig, (eixo_hist, eixo_box) = plt.subplots(
    2, 1, figsize=(8, 5), sharex=True, gridspec_kw={"height_ratios": [3, 1]}
)
eixo_hist.hist(W, bins=range(50_000, 800_001, 25_000), weights=[1 / len(W)] * len(W), color="#C44E52", edgecolor="white")
eixo_hist.set_ylabel("Frequência relativa")
eixo_hist.set_title("Distribuição de W em 1.365 imóveis com cozinha boa ou excelente")
eixo_box.boxplot(W, vert=False, widths=0.6)
eixo_box.set_yticks([])
eixo_box.set_xlabel("Preço de venda (W, US$ mil)")
eixo_box.xaxis.set_major_formatter(lambda valor, posicao: f"{valor / 1000:.0f}")
plt.show()

**Figura 3 - Distribuição de $W$.** A distribuição é unimodal, com pico na classe de US\$ 175 mil a US\$ 200 mil, e claramente assimétrica à direita (coeficiente 1,51): à esquerda, os preços terminam perto de US\$ 59 mil, enquanto à direita a cauda se estende até US\$ 755 mil. No boxplot, 51 imóveis ficam acima da cerca superior de Tukey (US\$ 414.250) e nenhum abaixo da inferior. O limite natural do preço é zero, que não chega a ser atingido, mas impede que a cauda esquerda acompanhe a direita.

**Síntese da caracterização empírica.** Nas três trilhas, as médias observadas servem de base para calibrar os parâmetros. Nas trilhas de contagem, as variâncias observadas superam em muito as previstas: em $X$, os grupos de 20 vendas variam quase três vezes mais do que o modelo Binomial prevê; em $Y$, a variância é cerca de 8 vezes a média, acompanhada de forte sazonalidade. Em $W$, o ponto central é a forma: o preço é positivo e assimétrico à direita, com cauda longa de imóveis de alto valor, enquanto o modelo Normal é simétrico e sem limites.

## 7. Modelo candidato e parâmetros

*Etapa 4 do enunciado: Modelo candidato e parâmetros (definição do modelo, determinação dos parâmetros, cálculo e interpretação da esperança e da variância).*

A **esperança** é a média teórica de uma variável aleatória, e a **variância** mede a dispersão teórica em torno dela. Em cada trilha, os parâmetros desconhecidos são determinados a partir dos dados observados, e a esperança e a variância do modelo assim calibrado são comparadas com a média e a variância observadas (Tabela 11).

### 7.1 Trilha Binomial

No modelo $X\sim\mathrm{Binomial}(n, p)$, a probabilidade de observar $k$ imóveis com cozinha boa ou excelente em $n$ vendas é

$$
P(X=k)=\binom{n}{k}p^k(1-p)^{n-k},\quad k=0, 1, \ldots, n.
$$

O número de ensaios é fixado pela construção dos grupos, $n=20$. A probabilidade de sucesso $p$ é determinada pela frequência relativa do evento nas 2.930 vendas, $\hat p = 1.365/2.930 \approx 0{,}466$. A esperança e a variância do modelo são

$$
\mathrm{E}(X)=np=20p
\quad\text{e}\quad
\mathrm{Var}(X)=np(1-p)=20p(1-p).
$$

In [ ]:
# Calcular a esperança e a variância de X, observadas e previstas pelo modelo Binomial
p = df["Kitchen Gd or Ex"].mean()

pd.DataFrame({
    "Observada": [X.mean(), X.var()],
    "Modelo Binomial(20, p)": [20 * p, 20 * p * (1 - p)],
}, index=["Esperança", "Variância"]).round(3)

**Tabela 13 - Esperança e variância de $X$.** Com $p \approx 0{,}466$, o modelo prevê, em média, 9,32 imóveis com cozinha boa ou excelente a cada 20 vendas, próximo da média observada de 9,35: o modelo localiza bem o centro da distribuição. A variância prevista, 4,98 (desvio padrão de cerca de 2,2 imóveis), significa que o modelo espera a grande maioria dos grupos entre cerca de 5 e 14 imóveis com o evento. A variância observada (13,57) é quase três vezes maior: os grupos variam muito mais do que ensaios independentes com probabilidade constante permitiriam.

### 7.2 Trilha Poisson

No modelo $Y\sim\mathrm{Poisson}(\lambda)$, a probabilidade de observar $k$ vendas com o evento em um mês é

$$
P(Y=k)=\frac{e^{-\lambda}\lambda^k}{k!},\quad k=0, 1, 2, \ldots
$$

O único parâmetro, $\lambda$, é a taxa média de vendas com o evento por mês, determinada pela média observada, $\hat\lambda = 1.365/55 \approx 24{,}82$. No modelo de Poisson, esperança e variância são iguais ao parâmetro:

$$
\mathrm{E}(Y)=\lambda
\quad\text{e}\quad
\mathrm{Var}(Y)=\lambda.
$$

In [ ]:
# Calcular a esperança e a variância de Y, observadas e previstas pelo modelo de Poisson
lam = Y.mean()

pd.DataFrame({
    "Observada": [Y.mean(), Y.var()],
    "Modelo Poisson(λ)": [lam, lam],
}, index=["Esperança", "Variância"]).round(3)

**Tabela 14 - Esperança e variância de $Y$.** Com $\lambda \approx 24{,}82$, o modelo prevê, em média, 24,82 vendas com o evento por mês, igual à média observada por construção. Como a variância é igual à média, o desvio padrão previsto é de cerca de 5 vendas ($\sqrt{24{,}82} \approx 4{,}98$), e o modelo espera a grande maioria dos meses entre cerca de 15 e 35 vendas com o evento. A variância observada (206,56) é cerca de 8 vezes a média, uma superdispersão coerente com a sazonalidade da Tabela 12.

### 7.3 Trilha Normal

No modelo $W\sim\mathrm{Normal}(\mu, \sigma^2)$, a densidade de probabilidade do preço de venda é

$$
f(w)=\frac{1}{\sigma\sqrt{2\pi}}\exp\left(-\frac{(w-\mu)^2}{2\sigma^2}\right),\quad -\infty<w<\infty.
$$

Os parâmetros são a própria esperança e a própria variância do modelo,

$$
\mathrm{E}(W)=\mu
\quad\text{e}\quad
\mathrm{Var}(W)=\sigma^2,
$$

e são determinados pela média e pela variância amostrais dos 1.365 preços.

In [ ]:
# Calcular a esperança e a variância de W, observadas e previstas pelo modelo Normal
mu = W.mean()
sigma2 = W.var()

pd.DataFrame({
    "Observada": [W.mean(), W.var()],
    "Modelo Normal(μ, σ²)": [mu, sigma2],
}, index=["Esperança", "Variância"]).style.format(precision=0)

**Tabela 15 - Esperança e variância de $W$.** Com $\mu \approx$ US\$ 229.834 e $\sigma^2 \approx 7{,}44 \times 10^9$ dólares², o modelo reproduz por construção a média e a variância observadas. Em termos interpretáveis, o desvio padrão é $\sigma \approx$ US\$ 86.229, isto é, 38% da média: os preços dos imóveis com cozinha boa ou excelente variam bastante em torno do valor típico.

In [ ]:
# Calcular o intervalo de dois desvios padrão em torno da média e a probabilidade de preço negativo no modelo Normal
sigma = sigma2 ** 0.5
pd.DataFrame({
    "Valor": [mu - 2 * sigma, mu + 2 * sigma, stats.norm.cdf(0, loc=mu, scale=sigma)],
}, index=["μ - 2σ (US$)", "μ + 2σ (US$)", "P(W < 0)"]).style.format(precision=4)

**Tabela 16 - Consequências dos parâmetros no modelo Normal.** O modelo espera cerca de 95% dos preços entre US\$ 57.377 e US\$ 402.292, faixa simétrica em torno de $\mu$. Como o desvio padrão é grande em relação à média, três desvios padrão abaixo de $\mu$ já ficam abaixo de zero, e o modelo atribui probabilidade de cerca de 0,4% a preços negativos, impossíveis na realidade. Além disso, a simetria do modelo põe a mediana sobre a média, enquanto a mediana observada (US\$ 213.000) fica abaixo dela, reflexo da assimetria à direita (Figura 3).

## 8. Conclusão e limitações

Esta etapa cobre os itens 1 a 4 do enunciado: delimitação da variável aleatória, compatibilidade inicial do modelo, descrição empírica e modelo candidato com seus parâmetros. A simulação, a auditoria dos pressupostos e o parecer de adequação (itens 5 a 7) ficaram fora do escopo, conforme orientação do professor. Por isso, a pergunta de pesquisa da Seção 2.2 não recebe aqui uma resposta definitiva, apenas os indícios levantados pela caracterização empírica.

- **Trilha Binomial.** A esperança prevista pelo modelo $\mathrm{Binomial}(20, p)$ (9,32) é praticamente igual à média observada (9,35), mas a variância observada (13,57) é quase três vezes a prevista (4,98). Os grupos de 20 vendas variam mais do que ensaios independentes com probabilidade constante permitiriam.
- **Trilha Poisson.** A média observada calibra $\lambda \approx 24{,}82$, mas a variância observada (206,56) é cerca de 8 vezes a média. A forte sazonalidade (Tabela 12) e a variação do total de vendas entre os meses (de 8 a 122) indicam que a taxa não é constante ao longo do tempo.
- **Trilha Normal.** Os parâmetros $\mu \approx$ US\$ 229.834 e $\sigma \approx$ US\$ 86.229 reproduzem a média e a variância observadas por construção, mas a distribuição dos preços é assimétrica à direita (coeficiente 1,51), com 51 imóveis acima da cerca superior de Tukey, e o modelo atribui probabilidade de cerca de 0,4% a preços negativos.

**Limitações.**

- **Escopo.** Sem os itens 5 a 7, as discrepâncias entre valores observados e previstos são registradas, mas não confrontadas formalmente com os pressupostos de cada modelo nem convertidas em parecer.
- **Construção dos grupos de $X$.** A ordenação por `Yr Sold`, `Mo Sold` e `Order` é cronológica apenas até o mês: dentro de cada mês, `Order` acompanha o código do imóvel (`PID`), que tende a reunir imóveis do mesmo bairro. Vendas consecutivas de um grupo podem, portanto, ser mais parecidas entre si do que vendas sorteadas ao acaso, o que contribui para a variância elevada de $X$.
- **Teto mensal de $Y$.** O número de vendas com o evento em um mês não pode superar o total de vendas desse mês, limite que o modelo de Poisson não contempla.
- **Calibração com a própria amostra.** Os parâmetros são estimados a partir dos mesmos dados com que os modelos são comparados, o que tende a aproximar as médias previstas das observadas.
- **Preços nominais.** $W$ usa preços nominais de 2006 a 2010, sem correção pela inflação nem pelas oscilações do mercado no período.
- **Período e local.** A base cobre apenas vendas residenciais em Ames, Iowa, entre 2006 e 2010, intervalo que inclui a crise financeira de 2008, e não deve ser generalizada para outras cidades ou períodos.

## 9. Contribuições dos integrantes

- **Adriel Medeiros Lins** (único integrante): definição do evento binário e das trilhas, preparação e agregação dos dados, cálculos, gráficos, interpretação dos resultados e redação do notebook.

## 10. Declaração de uso de inteligência artificial

TODO: ferramenta/modelo, finalidade, etapas afetadas e verificação do conteúdo (ou declaração de não uso).

## 11. Referências

DE COCK, Dean. Ames, Iowa: alternative to the Boston housing data as an end of semester regression project. *Journal of Statistics Education*, v. 19, n. 3, 2011. Disponível em: <https://jse.amstat.org/v19n3/decock.pdf>.

DE COCK, Dean. *Ames Housing dataset: data documentation* (`DataDocumentation.txt`). Dicionário oficial de variáveis e códigos, fornecido junto ao arquivo bruto (`data/raw/DataDocumentation.txt`).